In [153]:
# pip install scikit-learn

In [154]:
# pip install seaborn

In [155]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split



In [156]:
df = pd.read_excel('D:\Logistic _project\Data\Data_raw.xlsx')

In [157]:
mask = df['Transportation Distance (KM)'].isnull()

lat1 = np.radians(df.loc[mask, 'Origin Location Latitude'])
lon1 = np.radians(df.loc[mask, 'Origin Location Longitude'])
lat2 = np.radians(df.loc[mask, 'Destination Location Latitude'])
lon2 = np.radians(df.loc[mask, 'Destination Location Longitude'])

dphi = lat2 - lat1
dlambda = lon2 - lon1

a = np.sin(dphi/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlambda/2)**2
c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

r = 6371

df.loc[mask, 'Transportation Distance (KM)'] = r * c

In [158]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Bán kính Trái đất theo km
    r = 6371 
    
    # Chuyển đổi sang radian
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    # Công thức Haversine
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return r * c

# Giả sử df là DataFrame của bạn
# Bước 1: Xác định mask cho các dòng bị null
mask = df['Transportation Distance (KM)'].isnull()

# Bước 2: Chỉ tính toán cho những dòng bị thiếu để tối ưu tốc độ
# Lưu ý: Thêm .round(2) ở cuối để làm tròn 2 chữ số thập phân
df.loc[mask, 'Transportation Distance (KM)'] = df[mask].apply(
    lambda row: haversine_distance(
        row['Origin Location Latitude'], 
        row['Origin Location Longitude'], 
        row['Destination Location Latitude'], 
        row['Destination Location Longitude']
    ), axis=1
).round(0) # <--- Làm tròn ở đây

# Bước 3: Làm tròn toàn bộ cột một lần nữa để đảm bảo các giá trị cũ cũng sạch sẽ
df['Transportation Distance (KM)'] = df['Transportation Distance (KM)'].round(2)

# Nếu bạn muốn an toàn tuyệt đối với Power BI (không lo dấu phẩy/chấm), 
# hãy chuyển nó về số nguyên (Integer):
# df['Transportation Distance (KM)'] = df['Transportation Distance (KM)'].astype(int)

print(f"Đã xử lý xong. Số dòng còn trống: {df['Transportation Distance (KM)'].isnull().sum()}")

Đã xử lý xong. Số dòng còn trống: 0


In [159]:
df.isnull().sum()

Gps Provider                             0
Booking ID                               0
Shipment Type                            0
Booking Date                             0
Vehicle Registration                     0
Origin Location                          0
Destination Location                     0
Origin Location Latitude                 0
Origin Location Longitude                0
Destination Location Latitude            0
Destination Location Longitude           0
Data Ping time                           1
Planned ETA                              0
Current Location                         0
Actual ETA                              25
Current Location Latitude                1
Curren Location Longitude                1
Ontime                                   0
Trip Start Date                          0
Trip End Date                            0
Transportation Distance (KM)             0
Vehicle Type                           764
Minimum Kms To Be Covered In A Day    2645
Driver Name

In [160]:
date_cols = [
    'Booking Date',
    'Planned ETA',
    'Trip Start Date',
    'Data Ping time'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')



In [161]:
# =========================================
# 3. FEATURE ENGINEERING
# =========================================

# ----- Booking Date Features -----

df['booking_hour'] = df['Booking Date'].dt.hour
df['booking_dayofweek'] = df['Booking Date'].dt.dayofweek
df['booking_month'] = df['Booking Date'].dt.month

# ----- Planned ETA Features -----

df['planned_eta_hour'] = df['Planned ETA'].dt.hour
df['planned_eta_dayofweek'] = df['Planned ETA'].dt.dayofweek

# ----- Trip Start Features -----

df['trip_start_hour'] = df['Trip Start Date'].dt.hour

# ----- Ping Features -----

df['ping_hour'] = df['Data Ping time'].dt.hour

# =========================================
# 4. DROP LEAKAGE + UNUSED COLUMNS
# =========================================

drop_cols = [
    # 'Booking ID',
    'Vehicle Registration',
    'Current Location',
    'Driver Mobile No',
    'Actual ETA',
    'Trip End Date',
    'Customer Name',
    'Current Location Latitude',
    'Curren Location Longitude',
    'Booking Date',
    'Vehicle Type',
    'Minimum Kms To Be Covered In A Day',
    'Driver Name'
    
]

df = df.drop(columns=drop_cols, errors='ignore')

# =========================================
# 5. HANDLE TARGET
# =========================================

# Convert target to numeric
# Adjust if your dataset uses different values

df['Ontime'] = df['Ontime'].map({
    'Yes': 1,
    'No': 0,
    'YES': 1,
    'NO': 0,
    1: 1,
    0: 0
})

# Remove missing target rows
df = df.dropna(subset=['Ontime'])

# =========================================
# 6. DEFINE X AND y
# =========================================

X = df.drop(columns=['Ontime'])
y = df['Ontime']

In [162]:
# =========================================
# FEATURE ENGINEERING FOR LOCATION COLUMNS
# =========================================

# Fill missing values

df['Origin Location'] = df['Origin Location'].fillna('Unknown')
df['Destination Location'] = df['Destination Location'].fillna('Unknown')

# =========================================
# FUNCTION TO EXTRACT CITY & STATE
# =========================================

def extract_city_state(location):

    parts = [x.strip() for x in str(location).split(',')]

    # Example:
    # "Shive, Pune, Maharashtra"

    city = parts[-2] if len(parts) >= 2 else 'Unknown'

    state = parts[-1] if len(parts) >= 1 else 'Unknown'

    return pd.Series([city, state])

# =========================================
# ORIGIN FEATURES
# =========================================

df[['origin_city', 'origin_state']] = df[
    'Origin Location'
].apply(extract_city_state)

# =========================================
# DESTINATION FEATURES
# =========================================

df[['destination_city', 'destination_state']] = df[
    'Destination Location'
].apply(extract_city_state)

# =========================================
# SAME CITY FEATURE
# =========================================

df['same_city'] = (
    df['origin_city'].str.lower()
    ==
    df['destination_city'].str.lower()
).astype(int)

# =========================================
# SAME STATE FEATURE
# =========================================

df['same_state'] = (
    df['origin_state'].str.lower()
    ==
    df['destination_state'].str.lower()
).astype(int)

# =========================================
# KEEP ONLY NEEDED FEATURES
# =========================================

location_features = [
    'origin_state',
    'destination_state',
    'same_city',
    'same_state'
]

print(df[location_features].head())

  origin_state destination_state  same_city  same_state
0  Maharashtra        Tamil Nadu          0           0
1   Tamil Nadu        Tamil Nadu          1           1
2   Tamil Nadu        Tamil Nadu          1           1
3   Tamil Nadu        Tamil Nadu          1           1
4      Gujarat        Tamil Nadu          0           0


In [163]:
# =========================================
# FEATURE ENGINEERING FOR LOCATION COLUMNS
# =========================================

# Fill missing values

df['Origin Location'] = df['Origin Location'].fillna('Unknown')
df['Destination Location'] = df['Destination Location'].fillna('Unknown')

# =========================================
# FUNCTION TO EXTRACT CITY & STATE
# =========================================

def extract_city_state(location):

    parts = [x.strip() for x in str(location).split(',')]

    # Example:
    # "Shive, Pune, Maharashtra"

    city = parts[-2] if len(parts) >= 2 else 'Unknown'

    state = parts[-1] if len(parts) >= 1 else 'Unknown'

    return pd.Series([city, state])

# =========================================
# ORIGIN FEATURES
# =========================================

df[['origin_city', 'origin_state']] = df[
    'Origin Location'
].apply(extract_city_state)

# =========================================
# DESTINATION FEATURES
# =========================================

df[['destination_city', 'destination_state']] = df[
    'Destination Location'
].apply(extract_city_state)

# =========================================
# SAME CITY FEATURE
# =========================================

df['same_city'] = (
    df['origin_city'].str.lower()
    ==
    df['destination_city'].str.lower()
).astype(int)

# =========================================
# SAME STATE FEATURE
# =========================================

df['same_state'] = (
    df['origin_state'].str.lower()
    ==
    df['destination_state'].str.lower()
).astype(int)

# =========================================
# KEEP ONLY NEEDED FEATURES
# =========================================

location_features = [
    'origin_state',
    'destination_state',
    'same_city',
    'same_state'
]

print(df[location_features].head())

  origin_state destination_state  same_city  same_state
0  Maharashtra        Tamil Nadu          0           0
1   Tamil Nadu        Tamil Nadu          1           1
2   Tamil Nadu        Tamil Nadu          1           1
3   Tamil Nadu        Tamil Nadu          1           1
4      Gujarat        Tamil Nadu          0           0


In [164]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Gps Provider                    3585 non-null   object        
 1   Booking ID                      3585 non-null   object        
 2   Shipment Type                   3585 non-null   object        
 3   Origin Location                 3585 non-null   object        
 4   Destination Location            3585 non-null   object        
 5   Origin Location Latitude        3585 non-null   float64       
 6   Origin Location Longitude       3585 non-null   float64       
 7   Destination Location Latitude   3585 non-null   float64       
 8   Destination Location Longitude  3585 non-null   float64       
 9   Data Ping time                  3584 non-null   datetime64[ns]
 10  Planned ETA                     3585 non-null   datetime64[ns]
 11  Onti

In [165]:
df['trip_start_hour'] = df['Trip Start Date'].dt.hour
df['trip_start_dayofweek'] = (
    df['Trip Start Date'].dt.dayofweek
)

In [166]:
drop_cols = [
    'Origin Location',
    'Destination Location',
    'Origin Location Latitude',
    'Origin Location Longitude',
    'Destination Location Latitude',
    'Destination Location Longitude',
    'Data Ping time',
    'Planned ETA',
    'Material Shipped',
    'Gps Provider',
    'Supplier Name',
    'origin_city',
    'origin_state',
    'destination_city',
    'destination_state',
    'Trip Start Date'
]

df = df.drop(columns=drop_cols, errors='ignore')

In [167]:
df['Shipment Type'] = df['Shipment Type'].map({
    'Regular': 0,
    'Market': 1
})

In [168]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Booking ID                    3585 non-null   object 
 1   Shipment Type                 3585 non-null   int64  
 2   Ontime                        3585 non-null   int64  
 3   Transportation Distance (KM)  3585 non-null   float64
 4   booking_hour                  3585 non-null   int32  
 5   booking_dayofweek             3585 non-null   int32  
 6   booking_month                 3585 non-null   int32  
 7   planned_eta_hour              3585 non-null   int32  
 8   planned_eta_dayofweek         3585 non-null   int32  
 9   trip_start_hour               3585 non-null   int32  
 10  ping_hour                     3584 non-null   float64
 11  same_city                     3585 non-null   int64  
 12  same_state                    3585 non-null   int64  
 13  tri

In [169]:
# =========================================
# ONTIME PREDICTION WITH BOOKING ID TRACKING
# =========================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# =========================================
# 1. KEEP BOOKING ID FOR TRACKING
# =========================================

booking_ids = df['Booking ID']

# =========================================
# 2. DEFINE FEATURES AND TARGET
# =========================================

X = df.drop(columns=[
    'Booking ID',
    'Ontime'
])

y = df['Ontime']

# =========================================
# 3. TRAIN TEST SPLIT
# =========================================

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    booking_ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================
# 4. TRAIN MODEL
# =========================================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# =========================================
# 5. PREDICT
# =========================================

y_pred = model.predict(X_test)

# Probability of class = 1
y_prob = model.predict_proba(X_test)[:, 1]

# =========================================
# 6. EVALUATION
# =========================================

print("=" * 50)
print("ACCURACY")
print("=" * 50)

print(accuracy_score(y_test, y_pred))

print("\n")

print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)

print(classification_report(y_test, y_pred))

print("\n")

print("=" * 50)
print("CONFUSION MATRIX")
print("=" * 50)

print(confusion_matrix(y_test, y_pred))

# =========================================
# 7. CREATE PREDICTION RESULT TABLE
# =========================================

result_df = pd.DataFrame({

    'Booking ID': id_test,

    'Actual_Ontime': y_test,

    'Predicted_Ontime': y_pred,

    'Prediction_Probability': y_prob

})

# =========================================
# 8. RESET INDEX
# =========================================

result_df = result_df.reset_index(drop=True)

# =========================================
# 9. VIEW RESULTS
# =========================================

print("\n")
print("=" * 50)
print("PREDICTION RESULTS")
print("=" * 50)

print(result_df.head(10))

ACCURACY
0.898186889818689


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       441
           1       0.92      0.80      0.86       276

    accuracy                           0.90       717
   macro avg       0.90      0.88      0.89       717
weighted avg       0.90      0.90      0.90       717



CONFUSION MATRIX
[[423  18]
 [ 55 221]]


PREDICTION RESULTS
           Booking ID  Actual_Ontime  Predicted_Ontime  Prediction_Probability
0        AEIBK2021689              1                 0                0.121524
1        AEIBK2021630              0                 0                0.095195
2  VCV00010300/082021              1                 1                0.956843
3  MVCV0001108/082021              0                 1                0.643400
4        AEIBK2020520              1                 1                0.642902
5        AEIBK2024170              0                 0                0.084034
6     

In [171]:
# =========================================
# SHOW BOOKINGS PREDICTED AS DELAYED
# =========================================

# Assuming:
# 1 = Ontime
# 0 = Delayed

delayed_predictions = result_df[
    result_df['Predicted_Ontime'] == 0
]

# =========================================
# PRINT RESULT
# =========================================

print("=" * 50)
print("BOOKINGS PREDICTED AS DELAYED")
print("=" * 50)

print(delayed_predictions[[
    'Booking ID',
    'Prediction_Probability'
]])

# =========================================
# OPTIONAL: COUNT TOTAL DELAYS
# =========================================

print("\nTotal predicted delayed bookings:")

print(len(delayed_predictions))

BOOKINGS PREDICTED AS DELAYED
       Booking ID  Prediction_Probability
0    AEIBK2021689                0.121524
1    AEIBK2021630                0.095195
5    AEIBK2024170                0.084034
6    AEIBK2016157                0.004431
7    AEIBK2020963                0.125474
..            ...                     ...
709  AEIBK2021951                0.033392
710  AEIBK2025226                0.180217
711  AEIBK2021143                0.123096
712  AEIBK1910395                0.056143
716  AEIBK2022599                0.249828

[478 rows x 2 columns]

Total predicted delayed bookings:
478
